In [1]:
import json
import re
import os
import pathlib
import collections
from datetime import datetime

# Cấu hình đường dẫn
CONFIG_FILE = pathlib.Path('../data/documents_config.json')
INPUT_FILE = pathlib.Path('../data/du_lieu_phap_luat.json')
OUTPUT_DIR = pathlib.Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tải cấu hình văn bản (Config-driven)
documents_config = []
if CONFIG_FILE.exists():
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        documents_config = json.load(f)
    print(f"File cấu hình văn bản: {CONFIG_FILE.resolve()} (Đã tải {len(documents_config)} cấu hình)")
else:
    print(f"Cảnh báo: Không tìm thấy {CONFIG_FILE.resolve()}")

print(f"File đầu vào: {INPUT_FILE.resolve()}")
print(f"Thư mục đầu ra: {OUTPUT_DIR.resolve()}")

File cấu hình văn bản: D:\Đồ án tn\legal-ai-datn\backend\data\documents_config.json (Đã tải 5 cấu hình)
File đầu vào: D:\Đồ án tn\legal-ai-datn\backend\data\du_lieu_phap_luat.json
Thư mục đầu ra: D:\Đồ án tn\legal-ai-datn\backend\data\processed


## 1. Chuẩn Hóa Số Thứ Tự & Quy Ước Đặt ID (`node_id`)

Quy ước định danh cho từng cấp cấu trúc:
- `Document`: `{DOC_ID}` (ví dụ: `HIEN_PHAP_2013`, `BLDS_2015`)
- `Part` (Phần): `{DOC_ID}_P{n}` (ví dụ: `BLDS_2015_P1`)
- `Chapter` (Chương): `{DOC_ID}_CH{n}` hoặc `{DOC_ID}_P{p}_CH{n}`
- `Section` (Mục): `{DOC_ID}_CH{ch}_M{n}`
- `SubSection` (Tiểu mục): `{DOC_ID}_CH{ch}_M{m}_TM{n}` (ví dụ: `BLDS_2015_CH13_M1_TM1`)
- `Article` (Điều): `{DOC_ID}_D{n}` (ví dụ: `BLDS_2015_D584`, `HIEN_PHAP_2013_D1`)
- `Clause` (Khoản): `{DOC_ID}_D{dieu}_K{khoan}` (ví dụ: `BLDS_2015_D584_K1`)
- `Point` (Điểm): `{DOC_ID}_D{dieu}_K{khoan}_DIEM_{diem}` (ví dụ: `BLDS_2015_D584_K1_DIEM_A`)

> **Điểm mấu chốt:** SubSection được định danh gắn liền với parent context (`_CH{ch}_M{m}_TM{n}`) để đảm bảo 100% tính duy nhất toàn cục. Điểm dùng tiền tố `DIEM_` để tránh nhầm lẫn với Điều (`D`).


In [2]:
# Bảng ánh xạ số La Mã và từ thứ tự tiếng Việt sang số nguyên
VIETNAMESE_ORDINALS = {
    'nhất': 1, 'thứ nhất': 1, 'hai': 2, 'thứ hai': 2, 'ba': 3, 'thứ ba': 3,
    'bốn': 4, 'thứ tư': 4, 'thứ bốn': 4, 'tư': 4, 'năm': 5, 'thứ năm': 5,
    'sáu': 6, 'thứ sáu': 6, 'bảy': 7, 'thứ bảy': 7, 'tám': 8, 'thứ tám': 8,
    'chín': 9, 'thứ chín': 9, 'mười': 10, 'thứ mười': 10
}

ROMAN_NUMERALS = {
    'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6, 'VII': 7, 'VIII': 8, 'IX': 9, 'X': 10,
    'XI': 11, 'XII': 12, 'XIII': 13, 'XIV': 14, 'XV': 15, 'XVI': 16, 'XVII': 17, 'XVIII': 18, 'XIX': 19, 'XX': 20,
    'XXI': 21, 'XXII': 22, 'XXIII': 23, 'XXIV': 24, 'XXV': 25, 'XXVI': 26, 'XXVII': 27, 'XXVIII': 28, 'XXIX': 29, 'XXX': 30
}

def normalize_number(num_val):
    """Chuẩn hóa số nguyên, số La Mã hoặc từ thứ tự tiếng Việt về số nguyên."""
    if num_val is None:
        return ''
    s = str(num_val).strip()
    if s.isdigit():
        return int(s)
    s_lower = s.lower()
    if s_lower in VIETNAMESE_ORDINALS:
        return VIETNAMESE_ORDINALS[s_lower]
    s_upper = s.upper()
    if s_upper in ROMAN_NUMERALS:
        return ROMAN_NUMERALS[s_upper]
    return s

LOAI_TO_TYPE = {
    'van_ban': 'Document',
    'phan': 'Part',
    'chuong': 'Chapter',
    'muc': 'Section',
    'tieu_muc': 'SubSection',
    'dieu': 'Article',
    'khoan': 'Clause',
    'diem': 'Point',
}

CHILD_KEYS = ['cac_phan', 'cac_chuong', 'cac_muc', 'cac_tieu_muc', 'cac_dieu', 'cac_khoan', 'cac_diem', 'children']

def get_node_id(doc_id, item, context):
    """Sinh ID chuẩn hóa dựa trên loại node và ngữ cảnh cha con."""
    loai = item.get('loai')
    raw_so = item.get('so')
    norm_so = normalize_number(raw_so)
    
    if loai == 'van_ban':
        return doc_id
    elif loai == 'phan':
        return f"{doc_id}_P{norm_so}"
    elif loai == 'chuong':
        part_so = context.get('part_so')
        part_prefix = f"_P{part_so}" if part_so else ""
        return f"{doc_id}{part_prefix}_CH{norm_so}"
    elif loai == 'muc':
        chuong_so = context.get('chuong_so')
        ch_prefix = f"_CH{chuong_so}" if chuong_so else ""
        return f"{doc_id}{ch_prefix}_M{norm_so}"
    elif loai == 'tieu_muc':
        chuong_so = context.get('chuong_so')
        muc_so = context.get('muc_so')
        ch_prefix = f"_CH{chuong_so}" if chuong_so else ""
        m_prefix = f"_M{muc_so}" if muc_so else ""
        return f"{doc_id}{ch_prefix}{m_prefix}_TM{norm_so}"
    elif loai == 'dieu':
        return f"{doc_id}_D{norm_so}"
    elif loai == 'khoan':
        dieu_so = context.get('dieu_so')
        return f"{doc_id}_D{dieu_so}_K{norm_so}"
    elif loai == 'diem':
        dieu_so = context.get('dieu_so')
        khoan_so = context.get('khoan_so')
        diem_str = str(raw_so).strip().upper()
        if khoan_so:
            return f"{doc_id}_D{dieu_so}_K{khoan_so}_DIEM_{diem_str}"
        else:
            return f"{doc_id}_D{dieu_so}_DIEM_{diem_str}"
    else:
        return f"{doc_id}_{loai.upper()}_{norm_so}"
print("Đã tải thành công các hàm chuẩn hóa ID.")


Đã tải thành công các hàm chuẩn hóa ID.


## 2. Thuật Toán Duyệt Cây Đệ Quy Tổng Quát (Generic Recursive Traversal)

Thuật toán giải quyết triệt để vấn đề đa dạng cấu trúc giữa các văn bản pháp luật:
- **Không phụ thuộc độ sâu cố định:** Tự động phát hiện mọi danh sách con (`cac_phan`, `cac_chuong`, `cac_muc`, `cac_tieu_muc`, `cac_dieu`, `cac_khoan`, `cac_diem`, `children`).
- **Tự động gắn quan hệ cấu trúc:** Luôn tạo `(parent)-[:CONTAINS]->(child)` với `source_type = 'STRUCTURAL'`, `confidence = 1.0`.
- **Bảo tồn ngữ cảnh cha con:** Truyền số thứ tự của Phần, Chương, Điều, Khoản xuống các tầng con để phục vụ sinh ID.
- **Sinh `full_text` cho Article:** Tự động gộp Tiêu đề + Nội dung Điều + toàn bộ các Khoản và Điểm con thành trường `full_text` hoàn chỉnh, tối ưu cho bài toán Embedding/GraphRAG sau này.


In [3]:
def traverse_tree_recursive(item, parent_id, doc_id, context, nodes, contains_relations, seen_node_ids=None):
    """
    Duyệt cây đệ quy tổng quát (Generic Recursive Traversal)
    Tự động thích ứng với mọi độ sâu cấu trúc văn bản:
    - Bất kể văn bản có bao nhiêu tầng, hàm tự động kết nối (parent)-[:CONTAINS]->(child)
    - Tự động truyền ngữ cảnh phân cấp (part_so, chuong_so, dieu_so, khoan_so)
    """
    if seen_node_ids is None:
        seen_node_ids = set()
        
    loai = item.get('loai')
    if not loai:
        return
        
    node_type = LOAI_TO_TYPE.get(loai, loai.capitalize())
    base_node_id = get_node_id(doc_id, item, context)
    
    # Đảm bảo tính duy nhất tuyệt đối (V1)
    if loai == 'van_ban':
        current_node_id = doc_id
    else:
        current_node_id = base_node_id
        if current_node_id in seen_node_ids:
            dup_idx = 2
            while f"{base_node_id}_{dup_idx}" in seen_node_ids:
                dup_idx += 1
            current_node_id = f"{base_node_id}_{dup_idx}"
        seen_node_ids.add(current_node_id)
    
    # Cập nhật ngữ cảnh cho con cháu
    new_context = dict(context)
    norm_so = normalize_number(item.get('so'))
    if loai == 'phan':
        new_context['part_so'] = norm_so
    elif loai == 'chuong':
        new_context['chuong_so'] = norm_so
    elif loai == 'muc':
        new_context['muc_so'] = norm_so
    elif loai == 'dieu':
        new_context['dieu_so'] = norm_so
    elif loai == 'khoan':
        new_context['khoan_so'] = norm_so
        
    # Tạo node (loại trừ gốc van_ban đã được tạo trước)
    if loai != 'van_ban':
        title = item.get('ten') or item.get('tieu_de')
        text = item.get('noi_dung')
        number = item.get('so')
        
        node = {
            'node_id': current_node_id,
            'document_id': doc_id,
            'type': node_type,
            'label': node_type,
            'number': number,
            'title': title,
            'text': text,
            'parent_id': parent_id
        }
        if loai == 'dieu':
            full_parts = []
            if title:
                full_parts.append(title.strip())
            if text:
                full_parts.append(text.strip())
            for k in item.get('cac_khoan', []):
                k_so = k.get('so')
                k_noi_dung = k.get('noi_dung')
                if k_noi_dung:
                    full_parts.append(f'Khoản {k_so}. {k_noi_dung.strip()}')
                for p in k.get('cac_diem', []):
                    p_so = p.get('so')
                    p_noi_dung = p.get('noi_dung')
                    if p_noi_dung:
                        full_parts.append(f'Điểm {p_so}) {p_noi_dung.strip()}')
            node['full_text'] = '\n'.join(full_parts) if full_parts else None
            
        nodes.append(node)
        
        if parent_id:
            contains_relations.append({
                'source_id': parent_id,
                'type': 'CONTAINS',
                'target_id': current_node_id,
                'evidence': f"{parent_id} CONTAINS {current_node_id}",
                'source_type': 'STRUCTURAL',
                'confidence': 1.0
            })
            
    # Duyệt đệ quy tất cả danh sách con
    child_parent_id = current_node_id
    for k, v in item.items():
        if k in CHILD_KEYS or (k.startswith('cac_') and isinstance(v, list)):
            for child in v:
                if isinstance(child, dict) and 'loai' in child:
                    traverse_tree_recursive(child, child_parent_id, doc_id, new_context, nodes, contains_relations, seen_node_ids)
print("Đã tải hàm duyệt cây đệ quy tổng quát.")


Đã tải hàm duyệt cây đệ quy tổng quát.


## 3. Trích Xuất Dẫn Chiếu Pháp Lý (`REFERENCES`, `IMPLEMENTS`, `INTERPRETED_BY`) — Config-Driven & Hallucination Prevention

Nguyên tắc trích xuất quan hệ ngữ nghĩa pháp lý:
* **Chống tạo liên kết ảo (Hallucination Prevention):** Khi câu dẫn chiếu chỉ rõ văn bản ngoài corpus (ví dụ *"khoản 1 Điều 12 của Luật Nhà ở"*, *"khoản 18 Điều 3 của Luật Giao thông đường bộ"*...), bộ giải mã trả về `None` và ghi nhận vào danh sách `unresolved_references`, tuyệt đối **không tự ý fallback về văn bản hiện tại**.
* **Trích xuất toàn diện bao gồm cả Tiêu đề Điều:** Quét trích xuất trên `citation_text` gộp cả `title` và `text` của node nhằm bắt trọn các dẫn chiếu điều luật nằm trong tiêu đề Điều.
* **Chuẩn hóa chiều Ontology quan hệ cấp văn bản:**
  - `[:IMPLEMENTS]`: `[Nghị định/Thông tư thi hành]` $\rightarrow$ `[Luật/Bộ luật được thi hành]`.
  - `[:INTERPRETED_BY]`: `[Luật/Bộ luật được hướng dẫn]` $\rightarrow$ `[Nghị quyết hướng dẫn áp dụng]`.
* Mọi quan hệ đều có `evidence` ngữ cảnh chân thực, `source_type = 'EXPLICIT'`, `confidence = 1.0`.

In [4]:
"""
relation_parser.py
Module trích xuất quan hệ pháp lý với Config-driven Document Resolver & Target Specificity.
Cải tiến V2:
1. Chống False Edges (Hallucination Prevention): Nhận biết chính xác văn bản ngoài corpus và chuyển sang unresolved_references thay vì fallback sai.
2. Trích xuất cả Tiêu đề Điều (Article Title Citations): Bắt trọn vẹn dẫn chiếu điều luật trong tiêu đề.
3. Chuẩn hóa chiều quan hệ cấp văn bản (IMPLEMENTS, INTERPRETED_BY).
"""

import re
from typing import List, Dict, Any, Set, Optional, Tuple


def get_evidence(text: str, start: int, end: int, window: int = 35) -> str:
    """Trích xuất chuỗi bằng chứng ngữ cảnh bao quanh vị trí khớp Regex."""
    ctx_start = max(0, start - window)
    ctx_end = min(len(text), end + window)
    raw = text[ctx_start:ctx_end].strip()
    return f"...{raw}..." if (ctx_start > 0 or ctx_end < len(text)) else raw


class DocumentAliasRegistry:
    """
    Registry quản lý danh sách định danh (aliases) của các văn bản pháp luật,
    cho phép phân giải tự động từ ngữ cảnh trích dẫn sang document_id tương ứng
    hoàn toàn hướng cấu hình (Config-driven), ngăn chặn liên kết ảo.
    """
    def __init__(self, doc_configs: List[Dict[str, Any]] = None):
        self.doc_configs = doc_configs or []
        self.alias_to_doc: List[tuple] = []
        self.self_ref_terms: Dict[str, List[str]] = {
            'HIEN_PHAP': ['hiến pháp này'],
            'BO_LUAT': ['bộ luật này'],
            'LUAT': ['luật này'],
            'NGHI_DINH': ['nghị định này'],
            'THONG_TU': ['thông tư này'],
            'NGHI_QUYET': ['nghị quyết này'],
            'QUYET_DINH': ['quyết định này'],
            'PHAP_LENH': ['pháp lệnh này'],
        }
        self.doc_type_map: Dict[str, str] = {}
        self.build_registry()

    def build_registry(self):
        self.alias_to_doc = []
        for cfg in self.doc_configs:
            doc_id = cfg.get('document_id')
            if not doc_id:
                continue
            meta = cfg.get('metadata', {})
            doc_type = meta.get('loai_van_ban', '').upper()
            self.doc_type_map[doc_id] = doc_type

            aliases = list(cfg.get('aliases', []))
            so_hieu = meta.get('so_hieu')
            if so_hieu and so_hieu not in aliases:
                aliases.append(so_hieu)
            ten_vb = meta.get('ten_van_ban')
            if ten_vb and ten_vb not in aliases:
                aliases.append(ten_vb)

            for a in aliases:
                if a and a.strip():
                    self.alias_to_doc.append((a.strip().lower(), doc_id))

        # Sắp xếp alias giảm dần theo độ dài để ưu tiên cụm từ đặc thù dài nhất
        self.alias_to_doc.sort(key=lambda x: len(x[0]), reverse=True)

    def resolve(self, text: str, start: int, end: int, current_doc_id: str) -> Optional[str]:
        """
        Xác định document_id mục tiêu từ ngữ cảnh dẫn chiếu.
        Trả về:
          - doc_id nếu nhận diện được (trong corpus)
          - current_doc_id nếu là dẫn chiếu nội bộ
          - None nếu dẫn chiếu tường minh tới văn bản ngoài corpus (unresolved)
        """
        after_snippet = text[end: min(len(text), end + 70)]
        for stop_char in [';', '.\n', '. ']:
            if stop_char in after_snippet:
                after_snippet = after_snippet[:after_snippet.index(stop_char)]

        before_snippet = text[max(0, start - 40): start]
        match_snippet = text[start:end]

        cur_type = self.doc_type_map.get(current_doc_id, '')
        type_terms = self.self_ref_terms.get(cur_type, []) + ['văn bản này', 'quy định này']
        after_low = after_snippet.lower()
        
        candidates = []

        # 1. Kiểm tra self-references trong after_snippet
        for term in type_terms:
            pos = after_low.find(term)
            if pos != -1:
                candidates.append((pos, current_doc_id, term))

        # 2. Kiểm tra known aliases trong after_snippet
        for alias, doc_id in self.alias_to_doc:
            if alias == 'hiến pháp' and 'dự thảo' in after_low:
                continue
            pos = -1
            if len(alias) <= 4:
                m = re.search(r'\b' + re.escape(alias) + r'\b', after_low)
                if m:
                    pos = m.start()
            else:
                pos = after_low.find(alias)
            if pos != -1:
                candidates.append((pos, doc_id, alias))

        # 3. Kiểm tra văn bản bên ngoài corpus (External Law mentions)
        # e.g.: "của Luật Nhà ở", "của Luật Giao thông đường bộ", "của Bộ luật Tố tụng dân sự"
        ext_pattern = re.compile(
            r'(?:(?:của|thuộc|theo\s+(?:quy\s+định\s+(?:của|tại)\s+)?|trong)\s+)?\b(hiến pháp|bộ luật|luật|nghị định|nghị quyết|thông tư|pháp lệnh|quyết định)\s+([a-zà-ỹ0-9_\-\/]+)',
            re.IGNORECASE
        )
        for m in ext_pattern.finditer(after_low):
            following_word = m.group(2).lower()
            if following_word == 'này':
                continue
            matched_span = m.span()
            is_known = False
            for c_pos, c_doc, c_text in candidates:
                if c_doc != current_doc_id:
                    if (c_pos <= matched_span[0] and c_pos + len(c_text) >= matched_span[0]) or (matched_span[0] <= c_pos < matched_span[1]):
                        is_known = True
                        break
            if not is_known:
                candidates.append((m.start(), None, m.group(0)))

        # Sắp xếp theo vị trí gần nhất
        if candidates:
            candidates.sort(key=lambda x: x[0])
            best_pos, best_doc, best_term = candidates[0]
            if best_doc is None:
                return None  # Văn bản ngoài corpus -> Trả về None để tránh false edge
            return best_doc

        # Kiểm tra match_snippet
        match_low = match_snippet.lower()
        for alias, doc_id in self.alias_to_doc:
            if alias == 'hiến pháp' and 'dự thảo' in match_low:
                continue
            if len(alias) <= 4:
                if re.search(r'\b' + re.escape(alias) + r'\b', match_low):
                    return doc_id
            else:
                if alias in match_low:
                    return doc_id

        # Kiểm tra before_snippet
        before_low = before_snippet.lower()
        latest_pos = -1
        best_doc_before = None
        for alias, doc_id in self.alias_to_doc:
            if alias == 'hiến pháp' and 'dự thảo' in before_low:
                continue
            pos = -1
            if len(alias) <= 4:
                matches = list(re.finditer(r'\b' + re.escape(alias) + r'\b', before_low))
                if matches:
                    pos = matches[-1].start()
            else:
                pos = before_low.rfind(alias)
            if pos != -1 and pos > latest_pos:
                latest_pos = pos
                best_doc_before = doc_id

        if best_doc_before is not None:
            return best_doc_before

        return current_doc_id


DOCUMENT_REGISTRY = DocumentAliasRegistry(documents_config)


def resolve_citation_doc_id(text: str, start: int, end: int, current_doc_id: str, registry: DocumentAliasRegistry = None) -> Optional[str]:
    """Xác định document_id mục tiêu từ ngữ cảnh dẫn chiếu qua DocumentAliasRegistry."""
    reg = registry or DOCUMENT_REGISTRY
    return reg.resolve(text, start, end, current_doc_id)


def extract_references(
    nodes: List[Dict[str, Any]], 
    node_id_set: Set[str], 
    registry: DocumentAliasRegistry = None
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Trích xuất toàn bộ quan hệ dẫn chiếu pháp lý REFERENCES từ nội dung văn bản của các nodes.
    Quét trên cả title và text để bắt trọn dẫn chiếu trong tiêu đề Điều.
    """
    reg = registry or DOCUMENT_REGISTRY

    range_dieu_rx = re.compile(
        r'từ\s+[Đđ]iều\s+(\d+)\s+đến\s+(?:[Đđ]iều\s+)?(\d+)',
        re.IGNORECASE
    )
    list_diem_rx = re.compile(
        r'(?:[Đđ]iểm\s+([a-zđ])(?:\s*(?:,|và|hoặc)\s*(?:[Đđ]iểm\s+)?([a-zđ]))+)\s+(?:(?:của\s+)?[Kk]hoản\s+(\d+)\s+)?(?:của\s+)?[Đđ]iều\s+(\d+)',
        re.IGNORECASE
    )
    list_khoan_rx = re.compile(
        r'(?:[Kk]hoản\s+(\d+)(?:\s*(?:,|và|hoặc)\s*(?:[Kk]hoản\s+)?(\d+))+)\s+(?:của\s+)?[Đđ]iều\s+(\d+)',
        re.IGNORECASE
    )
    list_dieu_rx = re.compile(
        r'(?:(?:các\s+)?[Đđ]iều\s+(\d+)(?:\s*(?:,|và)\s*(?:[Đđ]iều\s+)?(\d+))+)',
        re.IGNORECASE
    )
    internal_dieu_rx = re.compile(
        r'(?:[Đđ]iểm\s+([a-zđ])(?:\s*(?:,|và|hoặc)\s*(?:[Đđ]iểm\s+)?([a-zđ]))*\s+)?(?:[Kk]hoản\s+(\d+)(?:\s*(?:,|và|hoặc)\s*(?:[Kk]hoản\s+)?(\d+))*\s+)?(?:của\s+)?[Đđ]iều\s+này',
        re.IGNORECASE
    )
    single_ref_rx = re.compile(
        r'(?:(?:[Đđ]iểm\s+([a-zđ]))\s+)?(?:(?:[Kk]hoản\s+(\d+))\s+)?(?:của\s+)?[Đđ]iều\s+(\d+)',
        re.IGNORECASE
    )

    all_refs = []
    unresolved_refs = []

    for node in nodes:
        # Quét trích xuất gộp cả title và text (đặc biệt các Điều có tiêu đề mang dẫn chiếu)
        citation_text = "\n".join(x for x in [node.get('title'), node.get('text')] if x)
        source_id = node.get('node_id')
        doc_id = node.get('document_id')
        if not citation_text or not source_id or not doc_id:
            continue
            
        cur_dieu_m = re.search(r'_D(\d+)', source_id)
        cur_dieu = cur_dieu_m.group(1) if cur_dieu_m else None

        # Pattern 1: Range Điều
        for match in range_dieu_rx.finditer(citation_text):
            start_d, end_d = int(match.group(1)), int(match.group(2))
            if start_d < end_d and (end_d - start_d) <= 50:
                target_doc = resolve_citation_doc_id(citation_text, match.start(), match.end(), doc_id, reg)
                ev = get_evidence(citation_text, match.start(), match.end())
                if not target_doc:
                    unresolved_refs.append({'source_id': source_id, 'citation': match.group(0), 'evidence': ev, 'reason': 'EXTERNAL_DOCUMENT'})
                    continue
                for d_num in range(start_d, end_d + 1):
                    target_id = f"{target_doc}_D{d_num}"
                    if target_id != source_id and target_id in node_id_set:
                        all_refs.append({
                            'source_id': source_id,
                            'type': 'REFERENCES',
                            'target_id': target_id,
                            'evidence': ev,
                            'source_type': 'EXPLICIT',
                            'confidence': 1.0
                        })

        # Pattern 2: List Điểm
        for match in list_diem_rx.finditer(citation_text):
            target_doc = resolve_citation_doc_id(citation_text, match.start(), match.end(), doc_id, reg)
            full_match_text = match.group(0)
            dieu_num = match.group(match.lastindex)
            khoan_num = match.group(match.lastindex - 1)
            diem_letters = re.findall(r'[Đđ]iểm\s+([a-zđ])', full_match_text)
            ev = get_evidence(citation_text, match.start(), match.end())
            if not target_doc:
                unresolved_refs.append({'source_id': source_id, 'citation': match.group(0), 'evidence': ev, 'reason': 'EXTERNAL_DOCUMENT'})
                continue
            
            for dl in diem_letters:
                candidates = []
                if khoan_num:
                    candidates.append(f"{target_doc}_D{dieu_num}_K{khoan_num}_DIEM_{dl.upper()}")
                candidates.append(f"{target_doc}_D{dieu_num}_DIEM_{dl.upper()}")
                if khoan_num:
                    candidates.append(f"{target_doc}_D{dieu_num}_K{khoan_num}")
                candidates.append(f"{target_doc}_D{dieu_num}")
                chosen = next((c for c in candidates if c in node_id_set), None)
                if chosen and chosen != source_id:
                    all_refs.append({
                        'source_id': source_id,
                        'type': 'REFERENCES',
                        'target_id': chosen,
                        'evidence': ev,
                        'source_type': 'EXPLICIT',
                        'confidence': 1.0
                    })

        # Pattern 3: List Khoản
        for match in list_khoan_rx.finditer(citation_text):
            target_doc = resolve_citation_doc_id(citation_text, match.start(), match.end(), doc_id, reg)
            full_match_text = match.group(0)
            dieu_match = re.search(r'[Đđ]iều\s+(\d+)', full_match_text)
            if not dieu_match:
                continue
            dieu_num = dieu_match.group(1)
            khoan_part = full_match_text[:dieu_match.start()]
            khoan_nums = re.findall(r'\b\d+\b', khoan_part)
            ev = get_evidence(citation_text, match.start(), match.end())
            if not target_doc:
                unresolved_refs.append({'source_id': source_id, 'citation': match.group(0), 'evidence': ev, 'reason': 'EXTERNAL_DOCUMENT'})
                continue
            
            for k in khoan_nums:
                candidates = [f"{target_doc}_D{dieu_num}_K{k}", f"{target_doc}_D{dieu_num}"]
                chosen = next((c for c in candidates if c in node_id_set), None)
                if chosen and chosen != source_id:
                    all_refs.append({
                        'source_id': source_id,
                        'type': 'REFERENCES',
                        'target_id': chosen,
                        'evidence': ev,
                        'source_type': 'EXPLICIT',
                        'confidence': 1.0
                    })

        # Pattern 4: List Điều
        for match in list_dieu_rx.finditer(citation_text):
            target_doc = resolve_citation_doc_id(citation_text, match.start(), match.end(), doc_id, reg)
            full_match_text = match.group(0)
            dieu_nums = re.findall(r'\b\d+\b', full_match_text)
            if len(dieu_nums) >= 2:
                ev = get_evidence(citation_text, match.start(), match.end())
                if not target_doc:
                    unresolved_refs.append({'source_id': source_id, 'citation': match.group(0), 'evidence': ev, 'reason': 'EXTERNAL_DOCUMENT'})
                    continue
                for d in dieu_nums:
                    target_id = f"{target_doc}_D{d}"
                    if target_id != source_id and target_id in node_id_set:
                        all_refs.append({
                            'source_id': source_id,
                            'type': 'REFERENCES',
                            'target_id': target_id,
                            'evidence': ev,
                            'source_type': 'EXPLICIT',
                            'confidence': 1.0
                        })

        # Pattern 5: Dẫn chiếu nội bộ Điều này
        if cur_dieu:
            for match in internal_dieu_rx.finditer(citation_text):
                full_m = match.group(0)
                k_match = re.findall(r'[Kk]hoản\s+(\d+)|(?:\s*,\s*|\s+và\s+)(\d+)', full_m[:full_m.lower().find('điều')])
                k_nums = [g1 or g2 for g1, g2 in k_match]
                
                diem_part = full_m[:full_m.lower().find('khoản')] if 'khoản' in full_m.lower() else full_m[:full_m.lower().find('điều')]
                d_match = re.findall(r'[Đđ]iểm\s+([a-zđ])|(?:\s*,\s*|\s+và\s+)([a-zđ])\b', diem_part)
                d_letters = [g1 or g2 for g1, g2 in d_match if (g1 or g2).lower() not in ['và', 'của', 'hoặc']]
                
                candidates = []
                if d_letters:
                    for dl in d_letters:
                        if k_nums:
                            for kn in k_nums:
                                candidates.append(f"{doc_id}_D{cur_dieu}_K{kn}_DIEM_{dl.upper()}")
                        else:
                            candidates.append(f"{doc_id}_D{cur_dieu}_DIEM_{dl.upper()}")
                elif k_nums:
                    for kn in k_nums:
                        candidates.append(f"{doc_id}_D{cur_dieu}_K{kn}")
                else:
                    candidates.append(f"{doc_id}_D{cur_dieu}")
                    
                ev = get_evidence(citation_text, match.start(), match.end())
                for cand in candidates:
                    if cand in node_id_set and cand != source_id:
                        all_refs.append({
                            'source_id': source_id,
                            'type': 'REFERENCES',
                            'target_id': cand,
                            'evidence': ev,
                            'source_type': 'EXPLICIT',
                            'confidence': 1.0
                        })

        # Pattern 6: Single Citation
        for match in single_ref_rx.finditer(citation_text):
            target_doc = resolve_citation_doc_id(citation_text, match.start(), match.end(), doc_id, reg)
            ev = get_evidence(citation_text, match.start(), match.end())
            if not target_doc:
                unresolved_refs.append({'source_id': source_id, 'citation': match.group(0), 'evidence': ev, 'reason': 'EXTERNAL_DOCUMENT'})
                continue
                
            diem, khoan, target_dieu = match.groups()
            candidates = []
            if diem and khoan:
                candidates.append(f"{target_doc}_D{target_dieu}_K{khoan}_DIEM_{diem.upper()}")
            elif diem:
                candidates.append(f"{target_doc}_D{target_dieu}_DIEM_{diem.upper()}")
            if khoan:
                candidates.append(f"{target_doc}_D{target_dieu}_K{khoan}")
            candidates.append(f"{target_doc}_D{target_dieu}")
            
            chosen = next((c for c in candidates if c in node_id_set), None)
            if chosen and chosen != source_id:
                all_refs.append({
                    'source_id': source_id,
                    'type': 'REFERENCES',
                    'target_id': chosen,
                    'evidence': ev,
                    'source_type': 'EXPLICIT',
                    'confidence': 1.0
                })

    # Khử trùng lặp và gộp evidence
    unique_refs = {}
    for ref in all_refs:
        key = (ref['source_id'], ref['type'], ref['target_id'])
        if key not in unique_refs:
            unique_refs[key] = ref
        else:
            existing_ev = unique_refs[key]['evidence']
            new_ev = ref['evidence']
            if new_ev not in existing_ev:
                unique_refs[key]['evidence'] = f"{existing_ev} | {new_ev}"
                
    # Lọc độ đặc hiệu mục tiêu
    filtered_refs = []
    for ref in unique_refs.values():
        tid = ref['target_id']
        sid = ref['source_id']
        is_article = bool(re.search(r'_D\d+$', tid))
        if is_article:
            has_more_specific = any(
                (r['source_id'] == sid and r['target_id'] != tid and r['target_id'].startswith(f"{tid}_"))
                for r in unique_refs.values()
            )
            if has_more_specific:
                continue
        filtered_refs.append(ref)

    return filtered_refs, unresolved_refs


def extract_document_level_relations(
    raw_data: List[Dict[str, Any]], 
    registry: DocumentAliasRegistry, 
    node_id_set: Set[str],
    doc_configs: List[Dict[str, Any]] = None
) -> List[Dict[str, Any]]:
    """
    Trích xuất các quan hệ cấp văn bản (Document-level relationships):
    - [:IMPLEMENTS]: Nghị định/Thông tư thi hành Luật/Bộ luật.
    - [:INTERPRETED_BY]: Luật/Bộ luật được hướng dẫn/giải thích bởi Nghị quyết.
    """
    doc_relations = []
    seen = set()
    
    # 1. Trích xuất từ khai báo cấu hình
    if doc_configs:
        for cfg in doc_configs:
            cfg_doc_id = cfg.get('document_id')
            for rel in cfg.get('relations', []):
                source_id = rel.get('source_id') or cfg_doc_id
                target_id = rel.get('target_id')
                rtype = rel.get('type', 'IMPLEMENTS')
                
                # Chuẩn hóa chiều quan hệ INTERPRETED_BY theo Ontology:
                # [Văn bản được giải thích] -[:INTERPRETED_BY]-> [Nghị quyết hướng dẫn]
                if rtype == 'INTERPRETED_BY' and not rel.get('source_id') and source_id == cfg_doc_id:
                    source_id, target_id = target_id, source_id
                
                if source_id in node_id_set and target_id in node_id_set and source_id != target_id:
                    key = (source_id, rtype, target_id)
                    if key not in seen:
                        seen.add(key)
                        doc_relations.append({
                            'source_id': source_id,
                            'type': rtype,
                            'target_id': target_id,
                            'evidence': rel.get('evidence', f"Khai báo cấu hình: {source_id} -> {target_id}"),
                            'source_type': rel.get('source_type', 'MANUAL'),
                            'confidence': float(rel.get('confidence', 1.0))
                        })

    # 2. Tự động trích xuất quan hệ IMPLEMENTS từ tiêu đề văn bản
    implements_rx = re.compile(
        r'(?:quy định chi tiết|quy định thi hành|hướng dẫn thi hành|thi hành)\s+(?:một số điều của\s+)?(.+)',
        re.IGNORECASE
    )
    
    for raw_doc in raw_data:
        meta = raw_doc.get('metadata', {})
        source_id = raw_doc.get('document_id') or meta.get('document_id')
        if not source_id or source_id not in node_id_set:
            continue
            
        ten_vb = meta.get('ten_van_ban', '')
        m = implements_rx.search(ten_vb)
        if m:
            matched_phrase = m.group(1).lower()
            for alias, target_id in registry.alias_to_doc:
                if target_id != source_id and target_id in node_id_set:
                    if alias in matched_phrase:
                        key = (source_id, 'IMPLEMENTS', target_id)
                        if key not in seen:
                            seen.add(key)
                            doc_relations.append({
                                'source_id': source_id,
                                'type': 'IMPLEMENTS',
                                'target_id': target_id,
                                'evidence': f"Văn bản quy định thi hành/chi tiết theo tiêu đề: \"{ten_vb}\"",
                                'source_type': 'EXPLICIT',
                                'confidence': 1.0
                            })
                        break
                        
    return doc_relations

print("Đã tải thành công DocumentAliasRegistry và hàm trích xuất quan hệ (có kiểm soát Unresolved References).")

Đã tải thành công DocumentAliasRegistry và hàm trích xuất quan hệ (có kiểm soát Unresolved References).


## 4. Bộ Kiểm Định Chất Lượng Đồ Thị (Validator V1 — V8)

Kiểm tra 8 điều kiện bắt buộc trước khi lưu file:
- **V1 [DUPLICATE_NODE_ID]:** Mọi `node_id` là duy nhất
- **V2 [ORPHAN_NODE]:** Mọi `parent_id` tồn tại trong nodes
- **V3 [DANGLING_SOURCE]:** `source_id` của mọi quan hệ tồn tại trong nodes
- **V4 [DANGLING_TARGET_STRUCTURAL]:** `target_id` của CONTAINS tồn tại trong nodes
- **V5 [WARNING_DANGLING_REF]:** `target_id` của REFERENCES tồn tại trong nodes (warning)
- **V6 [INVALID_DOCUMENT_ID]:** `document_id` trong mọi node tồn tại trong tập các Document node
- **V7 [INVALID_RELATION_TYPE]:** `type` của quan hệ nằm trong whitelist
- **V8 [MISSING_EVIDENCE]:** REFERENCES phải có `evidence` không rỗng


In [5]:
def validate_graph(nodes, relations):
    """Xác thực toàn bộ các quy tắc kiểm tra tính toàn vẹn của đồ thị (V1 — V8)."""
    errors = []
    warnings = []
    
    node_id_set = {n['node_id'] for n in nodes}
    doc_id_set = {n['node_id'] for n in nodes if n.get('type') == 'Document'}
    
    # V1: Unique node_ids
    node_id_counts = collections.Counter(n['node_id'] for n in nodes)
    for nid, c in node_id_counts.items():
        if c > 1:
            errors.append(f"V1 [DUPLICATE_NODE_ID]: {nid} xuất hiện {c} lần")
            
    # V2: parent_id tồn tại
    for n in nodes:
        pid = n.get('parent_id')
        if pid is not None and pid not in node_id_set:
            errors.append(f"V2 [ORPHAN_NODE]: {n['node_id']} có parent_id không tồn tại: {pid}")
            
    # V3: source_id tồn tại
    for r in relations:
        if r['source_id'] not in node_id_set:
            errors.append(f"V3 [DANGLING_SOURCE]: source_id '{r['source_id']}' không có trong nodes")
            
    # V4: target_id của STRUCTURAL (CONTAINS) tồn tại
    for r in relations:
        if r['type'] == 'CONTAINS' and r['target_id'] not in node_id_set:
            errors.append(f"V4 [DANGLING_TARGET_STRUCTURAL]: CONTAINS target_id '{r['target_id']}' không có trong nodes")
            
    # V5: target_id của REFERENCES và quan hệ ngữ nghĩa tồn tại
    for r in relations:
        if r['type'] in ('REFERENCES', 'IMPLEMENTS', 'INTERPRETED_BY') and r['target_id'] not in node_id_set:
            warnings.append(f"V5 [WARNING_DANGLING_REF]: {r['type']} target_id '{r['target_id']}' không có trong nodes")
            
    # V6: document_id hợp lệ
    for n in nodes:
        if n['document_id'] not in doc_id_set:
            errors.append(f"V6 [INVALID_DOCUMENT_ID]: node {n['node_id']} có document_id không hợp lệ: {n['document_id']}")
            
    # V7: relation_type trong whitelist
    REL_WHITELIST = {'CONTAINS', 'REFERENCES', 'IMPLEMENTS', 'INTERPRETED_BY', 'AMENDS', 'REPEALS', 'REPLACED_BY', 'RELATED_TO'}
    for r in relations:
        if r['type'] not in REL_WHITELIST:
            errors.append(f"V7 [INVALID_RELATION_TYPE]: relation có type lạ '{r['type']}'")
            
    # V8: evidence không rỗng cho quan hệ ngữ nghĩa
    for r in relations:
        if r['type'] in ('REFERENCES', 'IMPLEMENTS', 'INTERPRETED_BY'):
            if not r.get('evidence') or not str(r.get('evidence')).strip():
                errors.append(f"V8 [MISSING_EVIDENCE]: {r['type']} từ {r['source_id']} -> {r['target_id']} thiếu evidence")
                
    return errors, warnings
print("Đã tải bộ kiểm định chất lượng đồ thị (V1 — V8).")

Đã tải bộ kiểm định chất lượng đồ thị (V1 — V8).


## 5. Chạy Pipeline Xử Lý 


In [6]:
# 1. Đọc dữ liệu thô
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# 2. Khởi tạo Metadata văn bản pháp lý từ documents_config.json (Config-driven)
DOC_METADATA = {
    cfg['document_id']: {
        'document_id': cfg['document_id'],
        'title': cfg.get('metadata', {}).get('ten_van_ban'),
        'number': cfg.get('metadata', {}).get('so_hieu'),
        'type': cfg.get('metadata', {}).get('loai_van_ban', 'LUAT'),
        'issuer': cfg.get('metadata', {}).get('co_quan_ban_hanh'),
        'effective_from': cfg.get('metadata', {}).get('effective_from'),
        'effective_to': cfg.get('metadata', {}).get('effective_to'),
        'status': cfg.get('metadata', {}).get('status', 'CON_HIEU_LUC')
    }
    for cfg in documents_config if 'document_id' in cfg
}

# Cập nhật registry với cấu hình hiện tại
DOCUMENT_REGISTRY = DocumentAliasRegistry(documents_config)

all_nodes = []
all_contains_relations = []

for raw_doc in raw_data:
    meta = raw_doc.get('metadata', {})
    doc_id = raw_doc.get('document_id') or meta.get('document_id')
    if not doc_id:
        ten_vb = meta.get('ten_van_ban', '')
        resolved_id = None
        for alias, did in DOCUMENT_REGISTRY.alias_to_doc:
            if alias in ten_vb.lower():
                resolved_id = did
                break
        doc_id = resolved_id or f"DOC_{re.sub(r'[^a-zA-Z0-9]', '_', ten_vb)[:20]}"
        
    doc_info = DOC_METADATA.get(doc_id, {
        'document_id': doc_id,
        'title': meta.get('ten_van_ban'),
        'number': meta.get('so_hieu'),
        'type': meta.get('loai_van_ban', 'LUAT'),
        'issuer': meta.get('co_quan_ban_hanh'),
        'effective_from': meta.get('effective_from'),
        'effective_to': meta.get('effective_to'),
        'status': meta.get('status', 'CON_HIEU_LUC')
    })
    
    # Tạo Document Node
    all_nodes.append({
        'node_id': doc_id,
        'document_id': doc_id,
        'type': 'Document',
        'label': 'Document',
        'number': doc_info.get('number'),
        'title': doc_info.get('title'),
        'doc_type': doc_info.get('type'),
        'issuer': doc_info.get('issuer'),
        'effective_from': doc_info.get('effective_from'),
        'effective_to': doc_info.get('effective_to'),
        'status': doc_info.get('status'),
        'text': None,
        'parent_id': None
    })
    
    # Duyệt cấu trúc phân cấp bằng thuật toán đệ quy tổng quát
    cau_truc = raw_doc.get('cau_truc', {})
    traverse_tree_recursive(cau_truc, doc_id, doc_id, {}, all_nodes, all_contains_relations)

# Trích xuất các quan hệ REFERENCES (chống tạo false edge cho văn bản ngoài corpus)
node_id_set = {n['node_id'] for n in all_nodes}
all_reference_relations, unresolved_references = extract_references(all_nodes, node_id_set, DOCUMENT_REGISTRY)

# Trích xuất các quan hệ cấp văn bản (IMPLEMENTS, INTERPRETED_BY)
all_document_level_relations = extract_document_level_relations(
    raw_data, DOCUMENT_REGISTRY, node_id_set, documents_config
)

# Gộp tất cả quan hệ
all_relations = all_contains_relations + all_reference_relations + all_document_level_relations

# Kiểm định chất lượng toàn vẹn đồ thị
errors, warnings = validate_graph(all_nodes, all_relations)

# Ghi dữ liệu ra các file JSON chuẩn cho đồ thị
with open(OUTPUT_DIR / 'nodes.json', 'w', encoding='utf-8') as f:
    json.dump(all_nodes, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / 'relations.json', 'w', encoding='utf-8') as f:
    json.dump(all_relations, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / 'unresolved_references.json', 'w', encoding='utf-8') as f:
    json.dump(unresolved_references, f, ensure_ascii=False, indent=2)

rel_counts = collections.Counter(r['type'] for r in all_relations)
print(f"Đã lưu thành công các file đồ thị chuẩn tại: {OUTPUT_DIR.resolve()}")
print(f"  - Nodes: {len(all_nodes):,} ({len(DOC_METADATA)} văn bản)")
print(f"  - CONTAINS: {rel_counts.get('CONTAINS', 0):,}")
print(f"  - REFERENCES: {rel_counts.get('REFERENCES', 0):,}")
print(f"  - IMPLEMENTS: {rel_counts.get('IMPLEMENTS', 0):,}")
print(f"  - INTERPRETED_BY: {rel_counts.get('INTERPRETED_BY', 0):,}")
print(f"  - Tổng relations: {len(all_relations):,}")
print(f"  - Unresolved references (Ngoài corpus): {len(unresolved_references):,}")

Đã lưu thành công các file đồ thị chuẩn tại: D:\Đồ án tn\legal-ai-datn\backend\data\processed
  - Nodes: 3,472 (5 văn bản)
  - CONTAINS: 3,467
  - REFERENCES: 471
  - IMPLEMENTS: 2
  - INTERPRETED_BY: 1
  - Tổng relations: 3,941
  - Unresolved references (Ngoài corpus): 2


## 6. Thống Kê Chi Tiết & Báo Cáo Kiểm Định


In [7]:
print("=" * 68)
print("             BẢNG THỐNG KÊ CHI TIẾT NODES ĐỒ THỊ")
print("=" * 68)
doc_count = sum(1 for n in all_nodes if n.get('type') == 'Document')
print(f"Tổng số văn bản (Documents): {doc_count:,}")
print(f"Tổng số nodes toàn đồ thị  : {len(all_nodes):,}")
node_types = collections.Counter(n['type'] for n in all_nodes)
for t, count in node_types.most_common():
    print(f"  - {t:<15}: {count:,}")

print("\n" + "=" * 68)
print("           BẢNG THỐNG KÊ QUAN HỆ (RELATIONSHIPS)")
print("=" * 68)
print(f"Tổng số quan hệ: {len(all_relations):,}")
rel_types = collections.Counter(r['type'] for r in all_relations)
for t, count in rel_types.most_common():
    print(f"  - {t:<15}: {count:,}")

if unresolved_references:
    print(f"\n  * Dẫn chiếu ngoài corpus (đã chặn false edges): {len(unresolved_references):,}")
    for u in unresolved_references:
        print(f"    -> [{u['source_id']}] '{u['citation']}': {u['evidence'][:65]}...")

print("\n" + "=" * 68)
print("           KẾT QUẢ KIỂM ĐỊNH TÍNH TOÀN VẸN (V1 — V8)")
print("=" * 68)
if errors:
    print(f"[THẤT BẠI] Phát hiện {len(errors)} lỗi nghiêm trọng:")
    for e in errors:
        print(f"  x {e}")
else:
    print(">> ĐẠT CHUẨN 100%! Không có orphan node hay ID bị trùng.")

if warnings:
    print(f"\n[CẢNH BÁO] {len(warnings)} cảnh báo:")
    for w in warnings[:5]:
        print(f"  ! {w}")
else:
    print(">> 0 Cảnh báo! 100% target của quan hệ pháp lý tồn tại trong cơ sở dữ liệu.")

             BẢNG THỐNG KÊ CHI TIẾT NODES ĐỒ THỊ
Tổng số văn bản (Documents): 5
Tổng số nodes toàn đồ thị  : 3,472
  - Clause         : 1,940
  - Article        : 912
  - Point          : 487
  - Chapter        : 48
  - Section        : 43
  - SubSection     : 31
  - Part           : 6
  - Document       : 5

           BẢNG THỐNG KÊ QUAN HỆ (RELATIONSHIPS)
Tổng số quan hệ: 3,941
  - CONTAINS       : 3,467
  - REFERENCES     : 471
  - IMPLEMENTS     : 2
  - INTERPRETED_BY : 1

  * Dẫn chiếu ngoài corpus (đã chặn false edges): 2
    -> [NQ_02_2022_D2_K2_DIEM_A] 'khoản 1 Điều 12': ...C gây thiệt hại. Theo quy định tại khoản 1 Điều 12 của Luật Nh...
    -> [NQ_02_2022_D12_K1] 'khoản 18 Điều 3': ...loại xe tương tự theo quy định tại khoản 18 Điều 3 của Luật Gi...

           KẾT QUẢ KIỂM ĐỊNH TÍNH TOÀN VẸN (V1 — V8)
>> ĐẠT CHUẨN 100%! Không có orphan node hay ID bị trùng.
>> 0 Cảnh báo! 100% target của quan hệ pháp lý tồn tại trong cơ sở dữ liệu.


## 7. Kiểm Thử Khả Năng Thích Ứng Cấu Trúc Dị Biệt (Heterogeneous Structure Test)

Chứng minh thuật toán đệ quy chạy ổn định trên các cấu trúc phân cấp phi chuẩn:


In [8]:
# 1. Cấu trúc đầy đủ: Chương -> Mục -> Điều -> Khoản -> Điểm (Nghị quyết A)
mock_nq_a = {
    'loai': 'chuong',
    'so': 'I',
    'ten': 'CHƯƠNG THỬ NGHIỆM',
    'cac_muc': [{
        'loai': 'muc',
        'so': 1,
        'ten': 'Mục 1 Thử Nghiệm',
        'cac_dieu': [{
            'loai': 'dieu',
            'so': 1,
            'tieu_de': 'Điều 1 Thử Nghiệm',
            'cac_khoan': [{
                'loai': 'khoan',
                'so': 1,
                'noi_dung': 'Khoản 1',
                'cac_diem': [{'loai': 'diem', 'so': 'a', 'noi_dung': 'Điểm a'}]
            }]
        }]
    }]
}

# 2. Cấu trúc rút gọn: Điều -> Khoản -> Điểm trực tiếp (Nghị quyết B)
mock_nq_b = {
    'loai': 'dieu',
    'so': 10,
    'tieu_de': 'Điều 10 Rút Gọn',
    'cac_khoan': [{
        'loai': 'khoan',
        'so': 1,
        'noi_dung': 'Nội dung khoản 1',
        'cac_diem': [{'loai': 'diem', 'so': 'b', 'noi_dung': 'Điểm b'}]
    }]
}

test_nodes, test_rels = [], []
traverse_tree_recursive(mock_nq_a, 'TEST_NQ_A', 'TEST_NQ_A', {}, test_nodes, test_rels)
traverse_tree_recursive(mock_nq_b, 'TEST_NQ_B', 'TEST_NQ_B', {}, test_nodes, test_rels)

print(f"Thử nghiệm cấu trúc dị biệt thành công! Sinh {len(test_nodes)} nodes và {len(test_rels)} quan hệ CONTAINS.")
for n in test_nodes:
    print(f"  Node: {n['node_id']:<35} (Type: {n['type']}, Parent: {n['parent_id']})")

# 3. Kiểm thử Document Resolver (Config-driven & Chống False Edges)
print("\n--- Kiểm thử Document Resolver (Chống False Edges) ---")
test_cases = [
    ("theo quy định tại Điều 338 của Bộ luật Dân sự", "ND_21_2021", "BLDS_2015"),
    ("căn cứ Điều 15 Hiến pháp", "ND_21_2021", "HIEN_PHAP_2013"),
    ("theo quy định tại Nghị định số 21/2021/NĐ-CP", "NQ_02_2022", "ND_21_2021"),
    ("quy định tại Điều này của Nghị định này", "ND_21_2021", "ND_21_2021"),
    ("theo quy định tại khoản 1 Điều 12 của Luật Nhà ở năm 2014", "NQ_02_2022", None),
    ("theo quy định tại khoản 18 Điều 3 của Luật Giao thông đường bộ năm 2008", "NQ_02_2022", None),
]
for text_sample, cur_doc, expected in test_cases:
    # Tìm span của trích dẫn trong câu mẫu
    m = re.search(r'(?:[Kk]hoản\s+\d+\s+)?[Đđ]iều\s+\d+', text_sample)
    s, e = m.span() if m else (0, len(text_sample))
    resolved = DOCUMENT_REGISTRY.resolve(text_sample, s, e, cur_doc)
    status = "OK" if resolved == expected else "FAIL"
    print(f"  [{status}] '{text_sample}' -> {resolved} (Kỳ vọng: {expected})")

Thử nghiệm cấu trúc dị biệt thành công! Sinh 8 nodes và 8 quan hệ CONTAINS.
  Node: TEST_NQ_A_CH1                       (Type: Chapter, Parent: TEST_NQ_A)
  Node: TEST_NQ_A_CH1_M1                    (Type: Section, Parent: TEST_NQ_A_CH1)
  Node: TEST_NQ_A_D1                        (Type: Article, Parent: TEST_NQ_A_CH1_M1)
  Node: TEST_NQ_A_D1_K1                     (Type: Clause, Parent: TEST_NQ_A_D1)
  Node: TEST_NQ_A_D1_K1_DIEM_A              (Type: Point, Parent: TEST_NQ_A_D1_K1)
  Node: TEST_NQ_B_D10                       (Type: Article, Parent: TEST_NQ_B)
  Node: TEST_NQ_B_D10_K1                    (Type: Clause, Parent: TEST_NQ_B_D10)
  Node: TEST_NQ_B_D10_K1_DIEM_B             (Type: Point, Parent: TEST_NQ_B_D10_K1)

--- Kiểm thử Document Resolver (Chống False Edges) ---
  [OK] 'theo quy định tại Điều 338 của Bộ luật Dân sự' -> BLDS_2015 (Kỳ vọng: BLDS_2015)
  [OK] 'căn cứ Điều 15 Hiến pháp' -> HIEN_PHAP_2013 (Kỳ vọng: HIEN_PHAP_2013)
  [OK] 'theo quy định tại Nghị định số 21/20